Display the data available in the lakehouse

In [1]:
df=spark.read.option("multiline","true").json("Files/latest-news")
display(df)

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 61052c5b-835e-46c4-aafb-f89bd8bd09c2)

Use explode() to convert data lakehouse data into a readable format

In [2]:
from pyspark.sql.functions import explode, col, concat_ws

# 1. Read and explode like before
df_raw = spark.read.option("multiline","true").json("Files/latest-news")

df_articles = df_raw.select(explode(col("results")).alias("article"))

df_flat = df_articles.select(
    col("article.title").alias("Title"),
    col("article.link").alias("Link"),
    col("article.description").alias("Description"),
    col("article.pubDate").alias("Published_Date"),
    col("article.source_id").alias("Source"),
    col("article.country"),
    col("article.category"),
    concat_ws(", ", col("article.creator")).alias("Authors"),
    concat_ws(", ", col("article.keywords")).alias("Keywords")
)

# 2. Convert to list of JSON strings - SAME as your screenshot
json_list = df_flat.toJSON().collect()

# 3. Print any record - SAME as your screenshot
print(json_list[9]) # first article


StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 4, Finished, Available, Finished, False)

{"Title":"CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities","Link":"https://thecsrjournal.in/cisf-airport-sector-signs-mou-with-iit-ropar-enhance-cyber-security-capabilities/","Description":"The Central Industrial Security Force (CISF) Airport Sector has taken a significant step to enhance cyber security in the aviation industry by signing a Memorandum of Understanding (MoU) with the Indian Institute of Technology Ropar (IIT Ropar) on August 7. This agreement facilitates a specialised six-week residential Cyber Security Training Programme tailored for CISF personnel. [...] The post CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities appeared first on The CSR Journal .","Published_Date":"2026-08-08 21:02:56","Source":"thecsrjournal","country":["india"],"category":["top","technology"],"Authors":"nirali sethi","Keywords":"technology, cisf, mou, aviation, cybersecurity, aviation industry"}


Converting data into array

In [3]:
import json
news_loads=json.loads(json_list[9])
print(news_loads)

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 5, Finished, Available, Finished, False)

{'Title': 'CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities', 'Link': 'https://thecsrjournal.in/cisf-airport-sector-signs-mou-with-iit-ropar-enhance-cyber-security-capabilities/', 'Description': 'The Central Industrial Security Force (CISF) Airport Sector has taken a significant step to enhance cyber security in the aviation industry by signing a Memorandum of Understanding (MoU) with the Indian Institute of Technology Ropar (IIT Ropar) on August 7. This agreement facilitates a specialised six-week residential Cyber Security Training Programme tailored for CISF personnel. [...] The post CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities appeared first on The CSR Journal .', 'Published_Date': '2026-08-08 21:02:56', 'Source': 'thecsrjournal', 'country': ['india'], 'category': ['top', 'technology'], 'Authors': 'nirali sethi', 'Keywords': 'technology, cisf, mou, aviation, cybersecurity, aviation industry'}


Coverting data into dictionary(key,value) pair

In [4]:
for key, value in news_loads.items():
    print(f"{key}: {value}")

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 6, Finished, Available, Finished, False)

Title: CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities
Link: https://thecsrjournal.in/cisf-airport-sector-signs-mou-with-iit-ropar-enhance-cyber-security-capabilities/
Description: The Central Industrial Security Force (CISF) Airport Sector has taken a significant step to enhance cyber security in the aviation industry by signing a Memorandum of Understanding (MoU) with the Indian Institute of Technology Ropar (IIT Ropar) on August 7. This agreement facilitates a specialised six-week residential Cyber Security Training Programme tailored for CISF personnel. [...] The post CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities appeared first on The CSR Journal .
Published_Date: 2026-08-08 21:02:56
Source: thecsrjournal
country: ['india']
category: ['top', 'technology']
Authors: nirali sethi
Keywords: technology, cisf, mou, aviation, cybersecurity, aviation industry


This code loops through all NewsData.io articles, safely extracts Title, Description, Link etc into lists, and handles missing fields so it doesn't crash

In [5]:
import json

title = []
description = []
category = []
url = []
authors = []
source = []
country = []
keywords = []
datePublished = []

for json_str in json_list:
    try:
        article = json.loads(json_str)
        
        # Use .get() + default '' so it never breaks
        title.append(article.get("Title", ""))
        description.append(article.get("Description", ""))
        url.append(article.get("Link", ""))
        category.append(article.get("category", [])) # keep as list
        source.append(article.get("Source", ""))
        country.append(article.get("country", []))
        authors.append(article.get("Authors", ""))
        keywords.append(article.get("Keywords", ""))
        datePublished.append(article.get("Published_Date", ""))
        
    except Exception as e:
        print(f"Error processing JSON object: {e}")

print(f"Total articles processed: {len(title)}")

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 7, Finished, Available, Finished, False)

Total articles processed: 10


Try displaying a list

In [6]:
title

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 8, Finished, Available, Finished, False)

['Millions Of Galaxy Phones Are Weeks Away From Free Samsung Upgrade - Forbes',
 'Verizon outage map: Is the cell tower down in your area? Latest updates as company acknowledges voice service issue',
 'iOS 27: The easiest beta to forget about',
 'Samsung Says More Galaxy Z Flip Users Are Upgrading to Galaxy Z Fold 8',
 'Tricycle, oxygen cylinder provided to a differently-abled person',
 'IIT Bhubaneswar Concludes Two-Day National MDP on Human Capital and Purpose-Driven Governance',
 'OpenAI Acquires AI Presentation Startup NextSlide, Folding Team Into ChatGPT',
 'Motorola Moto Pad 70 Arrives With 5G And 10,200mAh Battery: Price, Specs And Features',
 '25 Years Of Dil Chahta Hai: Farhan Akhtar reveals why actors hesitated to sign the film: “Everyone wanted to do a love triangle” 25 : Bollywood News',
 'CISF Airport Sector Signs MoU With IIT Ropar to Enhance Cyber Security Capabilities']

Define schema and convert data into a tabular format

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# 1. Combine the lists
# Make sure these are the same list names from previous cell
data = list(zip(title, description, category, url, source, country, authors, keywords, datePublished))

# 2. Define schema for NewsData.io
schema = StructType([
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("category", ArrayType(StringType()), True),  # category is a list ['tech', 'top']
    StructField("link", StringType(), True),
    StructField("source", StringType(), True),
    StructField("country", ArrayType(StringType()), True),   # country is a list ['india']
    StructField("authors", StringType(), True),
    StructField("keywords", StringType(), True),
    StructField("published_date", StringType(), True)
])

# 3. Create DataFrame
df_cleaned = spark.createDataFrame(data, schema=schema)

# 4. Check it
display(df_cleaned)
print(f"Total rows: {df_cleaned.count()}")

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1feef576-04c4-4f88-a9e9-0c9840c014ac)

Total rows: 10


Convert the date format

In [8]:
from pyspark.sql.functions import to_date, date_format, to_timestamp

df_cleaned_final = df_cleaned \
    .withColumn("published_ts", to_timestamp("published_date", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("published_date_formatted", date_format("published_ts", "dd-MMM-yyyy"))

display(df_cleaned_final)

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8afabff-4922-46c6-b5a7-e107998d7884)

Writing final dataframe into lakehouse DB in delta format

In [9]:
# Step 1: Create the database if it doesn't exist
spark.sql("CREATE DATABASE IF NOT EXISTS newsdata_lake_db")

# Step 2: Now save your table
df_cleaned_final.write.format("delta").mode("overwrite").saveAsTable("newsdata_lake_db.news_raw")

StatementMeta(, fde35a38-a41c-47cf-a09c-2df899a8961d, 11, Finished, Available, Finished, False)